# 출처 파일럿 리뷰
**Validating the Fear & Greed Index with Web-Based Financial Sentiment: Correlation and Lead-Lag Analysis**

목표 기간은 **2017-01-01~2026-10-25**, 정렬 기준은 `America/New_York`이다.
2017-01-03, 2020-03-16, 2026-10-23 각각 최대 3문서로 접근과 필드를 점검했다.
이 노트북의 기본 동작은 **저장한 실제 결과 조회**이다. 전체 기간 수집을 실행하지 않는다.
뉴스와 Reddit은 별도 자료와 지수를 사용한다. 주 지수는 count, 연속 지수는 강건성 비교이다.


In [1]:
from pathlib import Path
import runpy
BASE=Path.cwd().resolve()
if not (BASE/"midterm paper.py").exists():
    BASE=BASE/"midterm paper"
pipeline=runpy.run_path(str(BASE/"midterm paper.py"))
read_csv=pipeline["common"].read_csv
import json
from IPython.display import display,Markdown
RESULTS=BASE/"results"/"source_pilot"
print("Target: 2017-01-01 ~ 2026-10-25 | America/New_York")

Target: 2017-01-01 ~ 2026-10-25 | America/New_York


## 1. 실제 요청 결과
`failed`, `credentials_missing`, `not_attempted_*`를 자료 0건과 구분한다. HTTP 요청 로그는 `http_log.jsonl`에 있다.

In [2]:
summary=read_csv(RESULTS/"pilot_summary.csv")
display(summary)
print("Actual HTTP requests:",len((RESULTS/"http_log.jsonl").read_text(encoding="utf-8").splitlines()))

,source,source_type,requested_start,requested_end,checked_at,n_raw,n_deduplicated,n_exact_timestamps,n_in_requested_et_date,status,note,error_type,error,policy_url
0,guardian_html,news,2026-10-23,2026-10-23,2026-10-26T12:28:05.887321+00:00,3,3,3.0,3.0,sample_obtained,NaN,NaN,NaN,NaN
1,guardian_html,news,2020-03-16,2020-03-16,2026-10-26T12:28:30.947023+00:00,3,3,3.0,3.0,sample_obtained,NaN,NaN,NaN,NaN
2,guardian_html,news,2017-01-03,2017-01-03,2026-10-26T12:28:54.774612+00:00,3,3,3.0,3.0,sample_obtained,NaN,NaN,NaN,NaN
3,guardian_api,news,2026-10-23,2026-10-23,2026-10-26T12:29:18.356129+00:00,0,0,NaN,NaN,credentials_missing,GUARDIAN_API_KEY 없음. 인증 실패를 유도하는 요청은 보내지 않음.,NaN,NaN,NaN
4,guardian_api,news,2020-03-16,2020-03-16,2026-10-26T12:29:18.360629+00:00,0,0,NaN,NaN,credentials_missing,GUARDIAN_API_KEY 없음. 인증 실패를 유도하는 요청은 보내지 않음.,NaN,NaN,NaN
5,guardian_api,news,2017-01-03,2017-01-03,2026-10-26T12:29:18.362129+00:00,0,0,NaN,NaN,credentials_missing,GUARDIAN_API_KEY 없음. 인증 실패를 유도하는 요청은 보내지 않음.,NaN,NaN,NaN
6,gdelt_doc,news,2026-10-23,2026-10-23,2026-10-26T12:29:18.363631+00:00,0,0,NaN,NaN,failed,NaN,ConnectionError,"('Connection aborted.', RemoteDisconnected('Re...",NaN
7,gdelt_doc,news,2020-03-16,2020-03-16,2026-10-26T12:29:32.530056+00:00,0,0,NaN,NaN,failed,NaN,AccessError,HTTP 429: https://api.gdeltproject.org/api/v2/...,NaN
8,gdelt_doc,news,2017-01-03,2017-01-03,2026-10-26T12:29:48.709721+00:00,0,0,NaN,NaN,not_attempted_after_host_denial,NaN,AccessError,앞선 접근 제한 이후 해당 호스트 요청 중단: api.gdeltproject.org,NaN
9,reddit_official_rfr,reddit,2017-01-03,2017-01-03,NaN,0,0,NaN,NaN,not_attempted_research_access_unconfirmed,"공식 연구 승인이 확인되지 않아 게시물 요청 안 함. 공식 제공 범위는 최근 5년,...",NaN,NaN,https://support.reddithelp.com/hc/en-us/articl...


Actual HTTP requests: 18


## 2. 수집 문서와 필드 점검
원문 headline+description, 발행 시각, URL, VADER 점수를 확인한다. 표본은 접근 검증용이며 일별 지수나 상관을 계산하기에 충분하지 않다. 미국시장 키워드 검사는 간단한 품질 신호일 뿐 최종 관련성 판정이 아니다.

In [3]:
audit=read_csv(RESULTS/"news_sample_audit.csv")
display(audit.reindex(columns=["pilot_source","requested_day","published_at","date_et","title_raw","description_raw","neg","neu","pos","compound","us_equity_keyword_hit","url"]))
display(read_csv(RESULTS/"news_sample_quality.csv"))

,pilot_source,requested_day,published_at,date_et,title_raw,description_raw,neg,neu,pos,compound,us_equity_keyword_hit,url
0,guardian_html,2017-01-03,2017-01-03 19:25:18+00:00,2017-01-03,British Airways cabin crew to go on strike nex...,"About 2,500 members of BA’s Heathrow-based mix...",0.288,0.712,0.000,-0.8360,False,https://www.theguardian.com/business/2017/jan/...
1,guardian_html,2017-01-03,2017-01-03 18:59:46+00:00,2017-01-03,Richard Cousins' departure from Tesco board is...,Senior independent director arrived just two y...,0.000,1.000,0.000,0.0000,False,https://www.theguardian.com/business/nils-prat...
2,guardian_html,2017-01-03,2017-01-03 18:26:50+00:00,2017-01-03,Richard Cousins makes shock departure from Tes...,Compass chief resigns as a non-executive direc...,0.170,0.830,0.000,-0.5994,False,https://www.theguardian.com/business/2017/jan/...
3,guardian_html,2020-03-16,2020-03-16 23:02:25+00:00,2020-03-16,Wall Street suffers worst day since 1987 as re...,Rolling coverage of the latest economic and fi...,0.312,0.634,0.054,-0.8910,True,https://www.theguardian.com/business/live/2020...
4,guardian_html,2020-03-16,2020-03-16 22:51:37+00:00,2020-03-16,"As recession looms, what can ministers do for ...","VAT cuts, higher welfare payments and job subs...",0.258,0.742,0.000,-0.6808,False,https://www.theguardian.com/business/2020/mar/...
5,guardian_html,2020-03-16,2020-03-16 20:31:18+00:00,2020-03-16,Dow suffers biggest-ever points loss as FTSE 1...,Sharp losses recorded despite US interest rate...,0.434,0.507,0.059,-0.9323,True,https://www.theguardian.com/business/2020/mar/...
6,guardian_html,2026-10-23,2026-10-23 16:48:28+00:00,2026-10-23,Oil price rises above $117 a barrel amid fears...,Brent crude increase follows Tehran threatenin...,0.386,0.556,0.058,-0.9186,False,https://www.theguardian.com/business/2026/oct/...
7,guardian_html,2026-10-23,2026-10-23 16:04:14+00:00,2026-10-23,Tell us: have your spending habits changed rec...,We would like to hear whether your spending ha...,0.000,0.889,0.111,0.3612,False,https://www.theguardian.com/business/2026/oct/...
8,guardian_html,2026-10-23,2026-10-23 14:39:21+00:00,2026-10-23,Oil hits five-month high as global stocks wobb...,Rolling coverage of the latest economic and fi...,0.123,0.775,0.102,0.0516,False,https://www.theguardian.com/business/live/2026...


,pilot_source,requested_day,n_docs,n_us_keyword_hits,n_missing_descriptions
0,guardian_html,2017-01-03,3,0,0
1,guardian_html,2020-03-16,3,2,0
2,guardian_html,2026-10-23,3,0,0


## 3. FGI 전체 기간 점검
값이 없는 평일에는 거래소 휴일이 포함될 수 있다. 실제 거래일 누락으로 단정하지 않으며 보간하지 않는다. 2021년 이전/이후 자료 계보를 구분한다.

In [4]:
print(json.dumps(json.loads((RESULTS/"fgi_coverage.json").read_text(encoding="utf-8")),ensure_ascii=False,indent=2))
display(read_csv(RESULTS/"fgi_annual_coverage.csv"))

{
  "status": "cached_archive_audited",
  "n_observations": 2443,
  "min_date": "2017-01-03",
  "max_date": "2026-10-23",
  "weekday_dates_without_value": 117,
  "era_counts": {
    "cnn_endpoint_via_third_party_archive": 1439,
    "pre_2021_frozen_multi_source_archive": 1004
  },
  "no_interpolation": true,
  "sha256_raw": "768d714cd0fa259ddd7e0cec1b049587fc09def90797e8d1b02fc57870a0c7cc",
  "path": "C:\\Users\\rhffl\\data_mining\\midterm paper\\data\\external\\fear_greed_2017-01-01_2026-10-25.csv",
  "note": "원응답 캐시 점검이며 최신 서버 재검증은 아님. 2021년 전후 자료 계보 차이는 공식 지수 방법론 변경 확정과 다름."
}


,year,n_observations,first_date,last_date
0,2017,251,2017-01-03,2017-12-29
1,2018,251,2018-01-02,2018-12-31
2,2019,252,2019-01-02,2019-12-31
3,2020,231,2020-01-02,2020-12-31
4,2021,252,2021-01-04,2021-12-31
5,2022,251,2022-01-03,2022-12-30
6,2023,250,2023-01-03,2023-12-29
7,2024,252,2024-01-02,2024-12-31
8,2025,250,2025-01-02,2025-12-31
9,2026,203,2026-01-02,2026-10-23


## 4. 출처 리뷰와 확대 전 결정

In [5]:
review=BASE/"SOURCE_PILOT_REVIEW.md"
display(Markdown(review.read_text(encoding="utf-8")))

# 출처 파일럿 리뷰 · 2026-10-26

**목표 기간: 2017-01-01~2026-10-25.** 기간을 임의로 줄이지 않았다.
실제 뉴스 검증 날짜는 2017-01-03, 2020-03-16, 2026-10-23이다.
전체 기간 수집은 시작하지 않았고, 아래 성공은 해당 날짜의 소규모 접근 성공만 뜻한다.

| 출처 | 실제 확인 결과 | 전체 기간 사용 판단 |
|---|---|---|
| Guardian 공개 Business 아카이브 | 세 날짜 각각 3건, 총 9건. 목록·기사 공개 메타데이터 HTTP 200. 제목·설명·발행 시각·URL 확보 | 유력 후보. 모든 연도의 연속성·수집량·미국시장 관련성은 미검증 |
| Guardian Content API | `GUARDIAN_API_KEY` 없음. 인증 요청 안 함 | 키 확보 후 검색어·태그를 고정한 추가 파일럿 권장 |
| GDELT DOC | 2026-10-23 요청은 `RemoteDisconnected`에 따른 `ConnectionError`. 2020-03-16 요청은 HTTP 429. 2017 요청은 앞선 제한에 따라 안 보냄 | 이번 파일럿 실패. 최근 접근 기간 제한과 발행 시각 부재도 장기 표본에 부적합 |
| FNSPID 공개 아카이브 | Hugging Face 파일 목록 HTTP 200. 파일 크기만 확인, 뉴스 행 다운로드 안 함 | 작성자 공시 1999–2023. 2017/2020 날짜별 행 미검증, 2026 제공 근거 없음 |
| Reddit 공식 RFR | 공식 안내 검토. 연구 승인 확인이 없어 게시물 요청 안 함 | 최근 5년·6개월 지연 조건이므로 요청한 전 기간을 충족하지 못함 |
| Reddit Arctic Shift | API 문서만 검토. 연구 이용 허용 근거가 확인되지 않아 게시물 요청 안 함 | 공개 접근과 연구 이용 승인을 구분. 현재 주 자료로 선택하지 않음 |

## 뉴스 9건의 품질

- 문서 단위는 **원문 제목 + 줄바꿈 + 짧은 설명**이다. 각 필드와 `raw_text`를 별도 보존했다.
- 9건 모두 시간대가 있는 실제 발행 시각이 있고 뉴욕 시간 변환 날짜가 요청 날짜와 일치했다.
- URL 중복 제거 전후 모두 9건이다. 과거 제목 수정·기사 삭제·다른 제목의 재작성 여부는 미검증이다.
- 기본 VADER의 `neg,neu,pos,compound`와 범주를 확인했다. 이 소표본으로 일별 지수·상관을 추정하지 않았다.
- 미국시장 사전 정의 키워드 검출은 **2017년 0/3, 2020년 2/3, 2026년 0/3**이다.
  간단한 품질 지표이며 나머지가 무관하다는 확정 판정은 아니다.
- 영국 기업·경제 뉴스가 많아 본 자료는 미국 주식시장·연준·Wall Street 등의 관련성 기준을 먼저 고정해야 한다.
- 목록에서 처음 찾은 3건이므로 대표 표본이 아니다. 실제 일별 수집량을 3건으로 추정하지 않는다.

## 접근·규모·재현성

**Guardian:** 공식 개발자 키는 비상업적 학생 연구를 지원하며 공시 한도는 초당 1회·일 500회이다.
키 없는 HTML 파일럿은 robots 규칙과 최소 6초 간격을 지켰다. 유료 본문·로그인·CAPTCHA는 우회하지 않았다.
공개 메타데이터의 제목·설명·발행 시각만 읽었다. 전체 HTML 수집의 허용량과 모든 날짜의 완전성은
아직 확인되지 않았다. 중단 없이 받은 파일럿 목록 3개와 기사 9개는 해당 시점의 접근 가능성을 보여준다.

**GDELT:** 429 이후 해당 호스트 요청을 중단했다. `seendate`는 발행이 아닌 최초 탐지 시각이다.
다른 뉴스의 실제 발행 시각과 같은 의미로 섞지 않는다.

**FNSPID:** 파일 목록의 `All_external.csv`는 5,731,397,037바이트(약 5.73 GB),
`nasdaq_exteral_data.csv`는 23,232,979,597바이트(약 23.23 GB)이다.
목록 확인을 뉴스 수집 성공으로 표시하지 않았다. 채택 시 청크/Parquet 변환이 필요하며
기사·종목별 중복, 시각의 의미, 출처별 권리·라이선스를 점검해야 한다.

**Reddit:** 공식 안내는 연구용 RFR을 지정하고 미승인 제3자 도구의 연구 이용을 제한한다.
승인에는 기관 소속·연구 계획·기관 후원·윤리 검토 자료 등이 요구된다.
현재 제공 범위는 최근 5년, 6개월 지연이다. 승인만 받아도 2017~2026-10 전체가 확보되는 구조는 아니다.
이번에는 Reddit 표본 수·날짜별 실제 커버리지를 검증하지 않았다.

## FGI 전체 기간 점검

기존 실제 원응답 캐시에서 **2,443개 관측치**를 추출했다.
첫 날짜는 **2017-01-03**, 마지막은 **2026-10-23**이다.
목표 경계 2017-01-01과 2026-10-25는 일요일이며 값을 만들어 넣지 않았다.

- 2021-02-01 이전: 다중 출처에서 합쳐진 고정 아카이브 **1,004개**.
- 이후: CNN 엔드포인트에서 갱신한 것으로 작성자가 설명하는 제3자 아카이브 **1,439개**.
- 값이 없는 평일은 **117일**이다. 거래소 휴일도 포함될 수 있어 모두 결측 거래일이라고 부르지 않는다.
  거래소 캘린더 대조는 확대 전에 남아 있다.
- 중복 날짜와 0~100 범위를 확인했고 보간·전일 값 채우기를 하지 않았다.
- 이 구분은 **자료 계보 차이**이며 공식 지수 방법론이 해당 날짜에 바뀌었다는 확정이 아니다.
  후속 분석은 전체 기간과 2021-02-01 이후를 별도로 비교할 수 있다.

파일은 `data/external/fear_greed_2017-01-01_2026-10-25.csv`이다.
연도별 건수·빈 평일 목록·요청 로그는 `results/source_pilot/`에 있다.
FGI 점검은 캐시를 재사용했으며 최신 서버 값을 다시 받은 것은 아니다.

## 확대 전 제안

1. **Guardian을 뉴스 1순위 후보**로 두고 미국시장 필터와 일별 수집량을 추가 검증한다.
   API 키가 있으면 날짜·태그·검색어를 고정하기 쉽다. HTML의 장기 허용 범위와 누락도 점검한다.
2. **Reddit은 확보됐다고 가정하지 않는다.** 공식 접근 범위와 별도 적법 자료 확보 가능성을 판단한다.
   뉴스 목표 기간과 Reddit 가용 기간은 각각 표시하며 공통 기간을 임의 확정하지 않는다.
3. 뉴스와 Reddit을 연·월별 별도 파티션에 두고 `WSI_News_count`, `WSI_Reddit_count`를 각각 만든다.
   연속 지수는 강건성 비교, 통합 지수는 아직 보류한다.
4. 실제 가용 연도 전체/선택 연도/패널 워드클라우드와 Parquet·청크 처리는 출처 검토 후 확장한다.
   S&P 500 수익률 OLS는 최종 탐색 단계로 남기고 Probit은 포함하지 않는다.

## 근거 문서

- [Guardian 개발자 키·한도](https://open-platform.theguardian.com/access/)
- [Guardian API 문서](https://open-platform.theguardian.com/documentation/)
- [FNSPID 작성자 설명](https://github.com/zGeneral/fnspid/blob/main/README.md)
- [GDELT 과거 접근 범위](https://github.com/alex9smith/gdelt-doc-api)
- [FGI 아카이브 계보](https://github.com/whit3rabbit/fear-greed-data)
- [Reddit 연구 이용 정책](https://support.reddithelp.com/hc/en-us/articles/14945211791892-Developer-Platform-Accessing-Reddit-Data)
- [Reddit RFR 조건·범위](https://support.reddithelp.com/hc/en-us/articles/49381918834964-Reddit-for-Researchers-Program)
- [Arctic Shift API 문서](https://github.com/ArthurHeitmann/arctic_shift/blob/master/api/README.md)


## 5. 필요할 때만 파일럿 재실행
아래 실행 플래그는 기본 False이다. 429/접근 거부 응답에서 해당 호스트 요청을 멈춘다. Guardian API 키는 환경변수로만 읽고 로그에서 숨긴다.

In [6]:
RERUN_PILOT=False
if RERUN_PILOT:
    cfg=pipeline["Config"](root=BASE,start="2017-01-01",end="2026-10-25")
    pipeline["run_stage"](cfg,"pilot")